# Statement Vectorization Example

This notebook demonstrates how to use the vectorization options in the statement preprocessing module.

## Available Vectorization Options

The `preprocess_statement()` function now supports:

- **`'none'`** (default): No vectorization, just clean text and features
- **`'tfidf'`**: TF-IDF weighted term frequencies
- **`'bigram'`**: Unweighted counts of unigrams and bigrams
- **`'binary'`**: Binary (0/1) presence indicators of unigrams

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, '../src')

from preprocessing.statement_ds import preprocess_statement, preprocess_statement_train_test

# Load sample data
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test_nolabel.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nFirst statement: {train_df['statement'].iloc[0][:100]}...")

Train shape: (8950, 8)
Test shape: (3836, 7)

First statement: China is in the South China Sea and (building)a military fortress the likes of which perhaps the wor...


## Example 1: No Vectorization (Default)

In [2]:
# Process without vectorization
result_no_vec = preprocess_statement(
    train_df.head(100),
    vectorizer_type='none',  # No vectorization
    verbose=True
)

print(f"\nOutput shape: {result_no_vec.shape}")
print(f"\nColumns: {result_no_vec.columns.tolist()}")
print(f"\nFirst cleaned statement: {result_no_vec['statement_clean'].iloc[0]}")

Total rows: 100
Empty statements after cleaning: 0
stopword_removal=False, stemmer=none, lemmatizer=none

Output shape: (100, 15)

Columns: ['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state_info', 'party_affiliation', 'statement_clean', 'statement_clean_char_len', 'statement_clean_word_count', 'statement_upper_ratio', 'statement_exclamation_count', 'statement_question_count', 'statement_clean_digit_ratio']

First cleaned statement: china is in the south china sea and building a military fortress the likes of which perhaps the world has not seen .


## Example 2: TF-IDF Vectorization

In [3]:
# Process with TF-IDF vectorization
result_tfidf = preprocess_statement(
    train_df.head(100),
    vectorizer_type='tfidf',
    vectorizer_max_features=50,  # Top 50 features
    vectorizer_min_df=2,  # Must appear in at least 2 documents
    verbose=True
)

print(f"\nOutput shape: {result_tfidf.shape}")

# Show TF-IDF feature columns
tfidf_cols = [col for col in result_tfidf.columns if col.startswith('statement_clean_vec_')]
print(f"\nNumber of TF-IDF features: {len(tfidf_cols)}")
print(f"Sample TF-IDF columns: {tfidf_cols[:5]}")

# Display a sample of the TF-IDF values
print(f"\nTF-IDF values for first document:")
print(result_tfidf[tfidf_cols].iloc[0].head(10))

Total rows: 100
Empty statements after cleaning: 0
stopword_removal=False, stemmer=none, lemmatizer=none
Vectorization: tfidf

Output shape: (100, 65)

Number of TF-IDF features: 50
Sample TF-IDF columns: ['statement_clean_vec_000', 'statement_clean_vec_an', 'statement_clean_vec_and', 'statement_clean_vec_any', 'statement_clean_vec_are']

TF-IDF values for first document:
statement_clean_vec_000       0.000000
statement_clean_vec_an        0.000000
statement_clean_vec_and       0.396657
statement_clean_vec_any       0.000000
statement_clean_vec_are       0.000000
statement_clean_vec_as        0.000000
statement_clean_vec_budget    0.000000
statement_clean_vec_by        0.000000
statement_clean_vec_for       0.000000
statement_clean_vec_from      0.000000
Name: 0, dtype: float64


## Example 3: Bigram Vectorization

In [4]:
# Process with bigram vectorization (unigrams + bigrams, count-based)
result_bigram = preprocess_statement(
    train_df.head(100),
    vectorizer_type='bigram',
    vectorizer_max_features=100,  # Top 100 features
    verbose=True
)

print(f"\nOutput shape: {result_bigram.shape}")

# Show bigram feature columns
bigram_cols = [col for col in result_bigram.columns if col.startswith('statement_clean_vec_')]
print(f"\nNumber of bigram features: {len(bigram_cols)}")
print(f"Sample bigram columns: {bigram_cols[:5]}")

# Display a sample of the bigram counts
print(f"\nBigram counts for first document:")
print(result_bigram[bigram_cols].iloc[0][result_bigram[bigram_cols].iloc[0] > 0])

Total rows: 100
Empty statements after cleaning: 0
stopword_removal=False, stemmer=none, lemmatizer=none
Vectorization: bigram

Output shape: (100, 115)

Number of bigram features: 100
Sample bigram columns: ['statement_clean_vec_000', 'statement_clean_vec_about', 'statement_clean_vec_administration', 'statement_clean_vec_an', 'statement_clean_vec_and']

Bigram counts for first document:
statement_clean_vec_and       1
statement_clean_vec_has       1
statement_clean_vec_in        1
statement_clean_vec_in the    1
statement_clean_vec_is        1
statement_clean_vec_of        1
statement_clean_vec_the       3
Name: 0, dtype: int64


## Example 4: Binary Vectorization

In [5]:
# Process with binary vectorization (0/1 presence of unigrams)
result_binary = preprocess_statement(
    train_df.head(100),
    vectorizer_type='binary',
    vectorizer_max_features=75,
    verbose=True
)

print(f"\nOutput shape: {result_binary.shape}")

# Show binary feature columns
binary_cols = [col for col in result_binary.columns if col.startswith('statement_clean_vec_')]
print(f"\nNumber of binary features: {len(binary_cols)}")

# Display a sample of the binary values
print(f"\nBinary presence for first document:")
print(result_binary[binary_cols].iloc[0].head(10))

Total rows: 100
Empty statements after cleaning: 0
stopword_removal=False, stemmer=none, lemmatizer=none
Vectorization: binary

Output shape: (100, 90)

Number of binary features: 75

Binary presence for first document:
statement_clean_vec_000             0
statement_clean_vec_an              0
statement_clean_vec_and             1
statement_clean_vec_any             0
statement_clean_vec_are             0
statement_clean_vec_as              0
statement_clean_vec_budget          0
statement_clean_vec_by              0
statement_clean_vec_can             0
statement_clean_vec_corporations    0
Name: 0, dtype: int64


## Example 5: Train-Test Split with Proper Vectorizer Fitting

When working with train/test splits, use `preprocess_statement_train_test()` to ensure the vectorizer is fitted **only on training data** and then applied to test data. This prevents data leakage.

In [6]:
# Process train and test with proper vectorizer handling
train_proc, test_proc = preprocess_statement_train_test(
    train_df.head(200),
    test_df.head(100),
    vectorizer_type='tfidf',
    vectorizer_max_features=50,
    verbose=True
)

print(f"\nTrain processed shape: {train_proc.shape}")
print(f"Test processed shape: {test_proc.shape}")

# Verify same features in both
train_vec_cols = [col for col in train_proc.columns if col.startswith('statement_clean_vec_')]
test_vec_cols = [col for col in test_proc.columns if col.startswith('statement_clean_vec_')]

print(f"\nTrain vectorization features: {len(train_vec_cols)}")
print(f"Test vectorization features: {len(test_vec_cols)}")
print(f"\nFeatures match: {set(train_vec_cols) == set(test_vec_cols)}")

Total rows: 200
Empty statements after cleaning: 0
stopword_removal=False, stemmer=none, lemmatizer=none
Vectorization: tfidf
Total rows: 200
Empty statements after cleaning: 0
stopword_removal=False, stemmer=none, lemmatizer=none
Vectorization: tfidf
Total rows: 100
Empty statements after cleaning: 0
stopword_removal=False, stemmer=none, lemmatizer=none
Vectorization: tfidf

Train processed shape: (200, 65)
Test processed shape: (100, 64)

Train vectorization features: 50
Test vectorization features: 50

Features match: True


## Example 6: Combining with Other Preprocessing Options

In [7]:
# Combine vectorization with stemming, stopword removal, and rare token features
result_combined = preprocess_statement(
    train_df.head(100),
    stemmer='porter',  # Use Porter stemming
    stopword_removal=True,  # Remove stopwords
    add_rare_token_features=True,  # Add rare token count features
    vectorizer_type='tfidf',  # Add TF-IDF vectorization
    vectorizer_max_features=40,
    verbose=True
)

print(f"\nOutput shape: {result_combined.shape}")

# Show what columns were created
new_cols = [col for col in result_combined.columns if col.startswith('statement')]
print(f"\nNew statement-related columns ({len(new_cols)}):")
for col in new_cols[:15]:
    print(f"  - {col}")

Total rows: 100
Empty statements after cleaning: 0
stopword_removal=True, stemmer=porter, lemmatizer=none
Vectorization: tfidf

Output shape: (100, 57)

New statement-related columns (50):
  - statement
  - statement_clean
  - statement_clean_char_len
  - statement_clean_word_count
  - statement_upper_ratio
  - statement_exclamation_count
  - statement_question_count
  - statement_clean_digit_ratio
  - statement_clean_rare_token_count
  - statement_clean_avg_token_freq
  - statement_clean_vec_000
  - statement_clean_vec_american
  - statement_clean_vec_budget
  - statement_clean_vec_clinton
  - statement_clean_vec_countri


## Vectorization Parameters Reference

### Main Parameters:

- **`vectorizer_type`** (str): One of `'none'`, `'tfidf'`, `'bigram'`, `'binary'`
- **`vectorizer_max_features`** (int or None): Maximum number of features. `None` = unlimited
- **`vectorizer_min_df`** (int): Minimum document frequency. Terms appearing in fewer documents are ignored
- **`vectorizer_max_df`** (float): Maximum document frequency as a ratio (0.0 to 1.0). Terms appearing in more than this ratio of documents are ignored

### Vectorizer Details:

| Type | Description | Output |
|------|-------------|--------|
| `'none'` | No vectorization | Only text features |
| `'tfidf'` | TF-IDF weights | Term frequency-inverse document frequency scores |
| `'bigram'` | Unigram + bigram counts | Raw frequency counts of 1-word and 2-word phrases |
| `'binary'` | Unigram presence | Binary (0/1) presence of 1-word terms |

### Feature Column Naming:

Vectorized features are named as: `{output_col}_vec_{term}`

For example, if `output_col='statement_clean'`, a TF-IDF feature for the word "president" would be named:
`statement_clean_vec_president`